# Principal Component Analysis (PCA)

---

## Overview

**PCA** is used in exploratory data analysis and for *dimensionality reduction* by projecting data onto directions of maximum variance.

## Steps

1. **Center the data:** $\tilde{X} = X - \bar{X}$

2. **Compute the covariance matrix:** $S = \dfrac{1}{N-1} \tilde{X}^\top \tilde{X}$

3. **Eigendecomposition:** $S \mathbf{v}_i = \lambda_i \mathbf{v}_i$

4. **Sort** eigenvectors by descending eigenvalue

5. **Project:** $Z = \tilde{X} V_k$ where $V_k$ contains the top $k$ eigenvectors

**Explained variance ratio** of component $i$:
$$r_i = \frac{\lambda_i}{\sum_j \lambda_j}$$

**Connection to SVD:** The eigenvectors of $S$ equal the right singular vectors of $\tilde{X}$: $\tilde{X} = U \Sigma V^\top$

---

**Dataset:** Gym Members Exercise Tracking (or Iris)  
**Task:** Reduce dimensions, visualize clusters, build a scree plot.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()

from rice_ml.unsupervised_learning import PCA
from rice_ml.preprocess import StandardScaler

In [ ]:
try:
    df = pd.read_csv('../../../data/gym_members_exercise_tracking.csv')
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    X = df[num_cols].dropna().values.astype(float)
    labels = None
    print(f'Loaded gym dataset: {X.shape}')
except FileNotFoundError:
    from sklearn.datasets import load_iris
    iris = load_iris()
    X, labels = iris.data, iris.target
    print('CSV not found. using iris dataset')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Scaled shape: {X_scaled.shape}')

## Scree Plot

A scree plot shows how much variance each principal component explains. we look for the "elbow" to choose $k$.

In [ ]:
pca_full = PCA(n_components=X_scaled.shape[1])
pca_full.fit(X_scaled)

per_var = np.round(pca_full.explained_variance_ratio_ * 100, 2)
cumulative = np.cumsum(per_var)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.bar(range(1, len(per_var) + 1), per_var, color='steelblue')
ax1.set_xlabel('Principal Component', fontsize=14)
ax1.set_ylabel('Explained Variance (%)', fontsize=14)
ax1.set_title('Scree Plot', fontsize=16)

ax2.plot(range(1, len(cumulative) + 1), cumulative, marker='o', color='salmon')
ax2.axhline(90, color='black', linestyle='--', label='90% threshold')
ax2.set_xlabel('Number of Components', fontsize=14)
ax2.set_ylabel('Cumulative Variance (%)', fontsize=14)
ax2.set_title('Cumulative Explained Variance', fontsize=16)
ax2.legend(fontsize=12)

plt.tight_layout()
plt.show()

n_90 = int(np.argmax(cumulative >= 90)) + 1
print(f'{n_90} components explain >= 90% of variance')

In [ ]:
# Project to 2D
pca_2d = PCA(n_components=2)
X_proj = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
if labels is not None:
    for cls, color in zip(np.unique(labels), ['red', 'lightseagreen', 'steelblue']):
        mask = labels == cls
        plt.scatter(X_proj[mask, 0], X_proj[mask, 1], c=color,
                    label=f'Class {cls}', alpha=0.7)
    plt.legend(fontsize=13)
else:
    plt.scatter(X_proj[:, 0], X_proj[:, 1], alpha=0.5, color='steelblue')

plt.xlabel(f'PC 1 ({per_var[0]:.1f}% variance)', fontsize=14)
plt.ylabel(f'PC 2 ({per_var[1]:.1f}% variance)', fontsize=14)
plt.title('PCA: 2D Projection', fontsize=18)
plt.show()

In [ ]:
# Reconstruction quality. how much information do we keep with n_90 components?
pca_k = PCA(n_components=min(n_90, X_scaled.shape[1]))
X_reconstructed = pca_k.inverse_transform(pca_k.fit_transform(X_scaled))

reconstruction_error = np.mean((X_scaled - X_reconstructed) ** 2)
print(f'Reconstruction MSE with {n_90} components: {reconstruction_error:.4f}')

## Interpretation

- The **scree plot** shows variance captured by each PC. we typically keep enough PCs to explain 90%+ of variance.
- PCA is **unsupervised**: it finds directions of maximum variance without using class labels.
- The 2D projection is useful for **visualizing high-dimensional data** and spotting natural clusters.